In [1]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.language_models.chat_models import  BaseChatModel # to create our own class
from langchain_core.messages import (
    BaseMessage, AIMessage, SystemMessage, HumanMessage, ToolMessage
)
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import BaseTool
from langchain_core.utils.function_calling import convert_to_openai_tool
import json
import time as _time

import pandas as pd
import requests
from requests.exceptions import RequestException
import uuid
from typing import Any, List, Optional, Sequence, Union, Callable


embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

/Users/keshavsharmaog/Downloads/swiss-law-agentic-retrieval/env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/keshavsharmaog/Downloads/swiss-law-agentic-retrieval/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/gt/zsnjtvpd3891ny4sp9lsvh5c0000gn/T/ipykernel_81101/3477111319.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import Huggi

In [31]:
import re

In [127]:
## mistral model
# tool config
TOOL_RESULT_MAX_CHARS = 2000
HISTORY_WINDOW = 8 # how many last messages to keep in context

# helper functions

def _merge_consecutive_roles(messages: List[dict]) -> List[dict]:
    """
    Mistral chat template requires strict user/assistant alternation.
    Merge any consecutive messages with the same role into one,
    joining their content with a newline separator.
    System messages are always kept at the front and not merged.
    """

    system_msg = [m for m in messages if m["role"] == "system"]
    non_system = [m for m in messages if m["role"] != "system"]
    merged: List[dict] = []

    for msg in non_system:
        if merged and merged[-1]["role"] == msg["role"]:
            merged[-1] = {
                "role": msg["role"],
                "content":merged[-1]["content"] + "\n\n" + msg["content"],
            }
        else:
            merged.append(msg)

    return system_msg + merged

def _parse_tool_call(text: str) -> tuple[list, str]:
    """
    Extract tool-call JSON from model output.
    Handles bare JSON and markdown-fenced ```json ... ``` blocks.
    Also handles escaped underscores (e.g. retriever\\_tool → retriever_tool).
    """
    # Unescape markdown-escaped underscores the model sometimes emits
    text_clean = text.replace("\\_", "_")

    candidates: list[tuple[str, str]] = []

    for m in re.finditer(r"```(?:json)?\s*(.*?)```", text_clean, re.DOTALL):
        candidates.append((m.group(1).strip(), m.group(0)))

    for m in re.finditer(r'\{[^{}]*"tool"\s*:[^{}]*\{[^{}]*\}[^{}]*\}', text_clean, re.DOTALL):
        candidates.append((m.group(0).strip(), m.group(0)))

    for raw, full_match in candidates:
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if "tool" not in parsed:
            continue
        args = parsed.get("tool_input") or parsed.get("arguments") or {}
        tool_calls = [{
            "name": parsed["tool"],
            "args": args,
            "id":   f"call_{uuid.uuid4().hex[:8]}",
            "type": "tool_call",
        }]
        return tool_calls, text_clean.replace(full_match, "").strip()

    return [], text



class ModalMistralChat(BaseChatModel):
    api_url: str
    temperature: float = 0.1
    max_new_tokens: int = 512
    top_p: float = 0.95
    system_prompt: str
    bound_tools: List[dict] = []
    max_retries: int = 3
    http_timeout: int = 600

    @property
    def _llm_type(self) -> str:
        return "modal_mistral_chat"

    def bind_tools(self,
                   tools: Sequence[Union[dict, type, Callable, BaseTool]],
                   **kwargs: Any
                   ) -> "ModalMistralChat":
        tool_schemas = [convert_to_openai_tool(t)["function"] for t in tools]
        return self.model_copy(update={"bound_tools": tool_schemas})

    def _build_messages(self, messages: List[BaseMessage]) -> List[dict]:
        """
        Convert LangChain messages → API dicts, then:
        1. Apply sliding window
        2. Merge consecutive same-role messages (Mistral requirement)
        3. Ensure first non-system msg is 'user' and roles strictly alternate
        """

        has_system = any(isinstance(m, SystemMessage) for m in messages)
        body: List[dict] = []

        if not has_system:
            body.append({
                "role": "system",
                "content": self.system_prompt,
            })

        for msg in messages:
            if isinstance(msg, SystemMessage):
                body.append({
                    "role": "system",
                    "content": str(msg.content),
                })
            elif isinstance(msg, HumanMessage):
                body.append({
                    "role": "user",
                    "content": str(msg.content),
                })
            elif isinstance(msg, AIMessage):
                content = msg.content if msg.content else ""
                if msg.tool_calls:
                    tc = msg.tool_calls[0]
                    content = json.dumps({"tool": tc["name"], "tool_input": tc["args"]}, ensure_ascii=False)
                body.append({
                    "role": "assistant",
                    "content": content,
                })
            elif isinstance(msg, ToolMessage):
                body.append({
                    "role": "user",
                    "content": f"Tool Result: \n {str(msg.content)}",
                })

        # sliding window on non - system messages
        system_part = [m for m in body if m["role"] == "system"]
        non_system = [m for m in body if m["role"] != "system"]
        windowed = non_system[-HISTORY_WINDOW:]

        # ensure window starts with a 'user' message (Mistral requirement)
        while windowed and windowed[0]["role"] != 'user':
            windowed.pop(0)

        combined = system_part + windowed

        # merge consecutive same - role messages -> Mistral alteration requirement
        merged = _merge_consecutive_roles(combined)

        # final safety: ensure strict user / assistant alteration
        final_system = [m for m in merged if m["role"] == "system"]
        final_conv = [m for m in merged if m["role"] != "system"]

        sanitized: List[dict] = []
        for msg in final_conv:
            if sanitized and sanitized[-1]["role"] == msg["role"]:
                # Merge into previous (shouldn't happen after _merge, but safety net)
                sanitized[-1]["content"] += "\n\n" + msg["content"]
            else:
                sanitized.append(dict(msg))

        # if conversation is empty  or starts with assistant, prepend a dummy user msg
        if not sanitized or sanitized[0]["role"] != "user":
            sanitized.insert(0, {"role": "user", "content": "Continue."})

        return final_system + sanitized

    def _generate(
        self,
        messages: list[BaseMessage],
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> ChatResult:
        api_messages = self._build_messages(messages)

        # ── Debug: log payload size ────────────────────────────────────────────
        total_chars = sum(len(m.get("content", "")) for m in api_messages)
        est_tokens  = total_chars // 3  # rough estimate: 1 token ≈ 3 chars
        print(f"→ POST to Modal  |  {len(api_messages)} msgs  |  {total_chars:,} chars  |  ~{est_tokens:,} tokens")

        payload = {
            "messages": api_messages,
            "max_new_tokens": self.max_new_tokens,
            "temperature": self.temperature,
            "top_p": self.top_p,
        }

        # retry with exponentational backoff
        last_err = None
        for attempt in range(1, self.max_retries + 1):
            try:
                response = requests.post(
                    self.api_url,
                    json=payload,
                    timeout=self.http_timeout,
                )
                if response.ok:
                    break
                # Server error (5xx) — retry
                try:
                    err_body = response.json()
                except Exception:
                    err_body = response.text[:500]
                print(f"[ModalMistralChat] HTTP {response.status_code} (attempt {attempt}/{self.max_retries}) — {err_body}")
                last_err = response
                if attempt < self.max_retries:
                    wait = 2 ** attempt  # 2s, 4s, 8s
                    print(f"  ⏳ Retrying in {wait}s…")
                    _time.sleep(wait)
            except RequestException as e:
                print(f"[ModalMistralChat] Network error (attempt {attempt}/{self.max_retries}) — {e}")
                last_err = e
                if attempt < self.max_retries:
                    wait = 2 ** attempt
                    print(f"  ⏳ Retrying in {wait}s…")
                    _time.sleep(wait)

        # If all retries failed, raise
        if isinstance(last_err, RequestException):
            raise last_err
        if last_err is not None and hasattr(last_err, 'raise_for_status'):
            if not last_err.ok:
                last_err.raise_for_status()

        text: str = response.json()["response"].strip()

        if stop:
            for s in stop:
                text = text.split(s)[0]

        tool_calls, text = _parse_tool_call(text) if self.bound_tools else ([], text)

        return ChatResult(
            generations=[
                ChatGeneration(message=AIMessage(content=text, tool_calls=tool_calls)),
            ]
        )

### qwen 2.5 7b
class ModalQwenChat(BaseChatModel):
    api_url: str
    temperature: float = 0.1
    max_new_tokens: int = 512
    top_p: float = 0.9
    system_prompt: str
    bound_tools: List[dict] = []
    max_retries: int = 3
    http_timeout: int = 600

    @property
    def _llm_type(self) -> str:
        return "modal_qwen_chat"

    def bind_tools(
        self,
        tools: Sequence[Union[dict, type, Callable, BaseTool]],
        **kwargs: Any
    ) -> "ModalQwenChat":
        tool_schemas = [convert_to_openai_tool(t)["function"] for t in tools]
        return self.model_copy(update={"bound_tools": tool_schemas})

    def _build_messages(self, messages: List[BaseMessage]) -> List[dict]:
        """
        Qwen is more flexible than Mistral:
        - No strict alternation required
        - No merging required
        - But we still keep sliding window
        """

        has_system = any(isinstance(m, SystemMessage) for m in messages)
        body: List[dict] = []

        if not has_system:
            body.append({
                "role": "system",
                "content": self.system_prompt,
            })

        for msg in messages:
            if isinstance(msg, SystemMessage):
                body.append({"role": "system", "content": str(msg.content)})

            elif isinstance(msg, HumanMessage):
                body[0]["content"] += "Legal Case:\n\n" + str(msg.content)  # append to system prompt

            # elif isinstance(msg, AIMessage):
            #     content = msg.content if msg.content else ""
            #     if msg.tool_calls:
            #         tc = msg.tool_calls[0]
            #         content = json.dumps({
            #             "tool": tc["name"],
            #             "tool_input": tc["args"]
            #         }, ensure_ascii=False)

            #     body.append({"role": "assistant", "content": content})

            # elif isinstance(msg, ToolMessage):
            #     body.append({
            #         "role": "user",
            #         "content": f"Tool Result:\n{str(msg.content)}"
            #     })

        # sliding window (same as before)
        system_part = [m for m in body if m["role"] == "system"]
        non_system = [m for m in body if m["role"] != "system"]
        windowed = non_system[-HISTORY_WINDOW:]

        return system_part + windowed

    def _generate(
        self,
        messages: list[BaseMessage],
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> ChatResult:

        api_messages = self._build_messages(messages)
        # Debug info
        total_chars = sum(len(m.get("content", "")) for m in api_messages)
        est_tokens  = total_chars // 3
        print(f"→ POST to Modal (Qwen) | {len(api_messages)} msgs | {total_chars:,} chars | ~{est_tokens:,} tokens")

        payload = {
            "messages": api_messages,
            "max_new_tokens": self.max_new_tokens,
            "temperature": self.temperature,
            "top_p": self.top_p,
        }

        last_err = None

        for attempt in range(1, self.max_retries + 1):
            try:
                response = requests.post(
                    self.api_url,
                    json=payload,
                    timeout=self.http_timeout,
                )

                if response.ok:
                    break

                try:
                    err_body = response.json()
                except Exception:
                    err_body = response.text[:500]

                print(f"[ModalQwenChat] HTTP {response.status_code} (attempt {attempt}/{self.max_retries}) — {err_body}")
                last_err = response

                if attempt < self.max_retries:
                    wait = 2 ** attempt
                    print(f"⏳ Retrying in {wait}s…")
                    _time.sleep(wait)

            except RequestException as e:
                print(f"[ModalQwenChat] Network error (attempt {attempt}/{self.max_retries}) — {e}")
                last_err = e

                if attempt < self.max_retries:
                    wait = 2 ** attempt
                    print(f"⏳ Retrying in {wait}s…")
                    _time.sleep(wait)

        if isinstance(last_err, RequestException):
            raise last_err

        if last_err is not None and hasattr(last_err, "raise_for_status"):
            if not last_err.ok:
                last_err.raise_for_status()

        text: str = response.json()["response"].strip()

        if stop:
            for s in stop:
                text = text.split(s)[0]

        tool_calls, text = _parse_tool_call(text) if self.bound_tools else ([], text)

        return ChatResult(
            generations=[
                ChatGeneration(
                    message=AIMessage(content=text, tool_calls=tool_calls)
                )
            ]
        )
    
    
vectorstore = FAISS.load_local(
    "../faiss_index_new",
    embeddings,
    allow_dangerous_deserialization=True
)
train_csv = pd.read_csv("../data/train.csv")


### now retrieving using the llm

In [ ]:
QUERIES_GENERATE_SYSTEM_PROMPT = """You are NOT a legal advisor.

You MUST NOT answer the legal question.

Your ONLY task is to generate search queries for retrieving relevant Swiss law texts.

Given a legal case description, generate exactly 5 short German search queries.

PROCESS:

1. Identify core legal concepts (e.g., Grundbuch, Löschung, ungerechtfertigter Eintrag)
2. Convert them into phrases that match statutory wording

STRICT RULES:

* DO NOT answer the question

* DO NOT suggest legal remedies

* DO NOT explain anything

* DO NOT write sentences

* ONLY output search queries

* Each query must be short (keywords, not sentences)

* Queries must resemble wording used in laws

* STRICTLY AVOID:

  * case-specific facts (names, story details)
  * words like "Rechtsbehelfe", "Ansprüche", "könnte", "sollte"

OUTPUT FORMAT:

* Exactly 5 lines
* Each line = one query
* No numbering
* No symbols
* No extra text

"""

retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs={"k": 200},
)


In [19]:
laws_de = pd.read_csv("../data/laws_de.csv")

In [20]:
laws_de.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


### reasearch for query 1

In [109]:
query = train_csv.iloc[2]["query"]
gold_citations = train_csv.iloc[2]["gold_citations"].split(";")

In [110]:
query

'Das Kompetenzzentrum Völkerstrafrecht bei der Bundesanwaltschaft erhält einen Tipp von syrischen Flüchtlingen. In einer Gemeinschaftsunterkunft wurde Anwar R. entdeckt, der in leitender Funktion in einem Gefängnis des syrischen Luftwaffengeheimdienstes in Damaskus zwischen 2011 und 2012 für die brutale Folter von mindestens 4000 Menschen verantwortlich gewesen sein soll. Nachprüfungen ergeben, dass die Vorwürfe zutreffen könnten. Es finden sich Hinweise auf systematische, brutale, physische und psychische Misshandlungen; u.a. Schläge mit Stöcken, Kabeln und Peitschen sowie Elektroschocks. Der Geheimdienst wollte dadurch Regimegegner einschüchtern und Informationen erzwingen. Mindestens 58 Gefangene sind vermutlich sogar an den Folgen der Folter gestorben. Anwar R. war für die Organisation der Verhöre und der Vernehmungsverfahren sowie die Einteilung der Vernehmer verantwortlich. \nAls zuständige Staatsanwältin sollen Sie prüfen, ob das Verhalten einen Tatbestand des Römer Statuts verl

In [111]:
gold_citations

['Art. 264m StGB']

In [94]:
# checking each citation in laws_de to get the text of the law
for citation in gold_citations:
    matches = laws_de[laws_de["citation"] == citation]
    if not matches.empty:
        print(f"Citation: {citation}")
        for _, row in matches.iterrows():
            print(f"Title: {row['title']}")
            print(f"Text: {row['text'][:500]}")  # print first 500 chars of the law text
            print("-" * 80)
    else:
        print(f"Citation {citation} not found in laws_de.csv")

Citation Art. 975 ZGB not found in laws_de.csv
Citation: Art. 974 Abs. 2 ZGB
Title: Schweizerisches Zivilgesetzbuch vom 10. Dezember 1907 - 3.  Gegenüber bösgläubigen Dritten
Text: 2 Ungerechtfertigt ist der Eintrag, der ohne Rechtsgrund oder aus einem unverbindlichen Rechtsgeschäft erfolgt ist.
--------------------------------------------------------------------------------
Citation Art. 973 ZGB not found in laws_de.csv
Citation: Art. 661 ZGB
Title: Schweizerisches Zivilgesetzbuch vom 10. Dezember 1907 - a.  Ordentliche Ersitzung
Text: Ist jemand ungerechtfertigt im Grundbuch als Eigentümer eingetragen, so kann sein Eigentum, nachdem er das Grundstück in gutem Glauben zehn Jahre lang ununterbrochen und unangefochten besessen hat, nicht mehr angefochten werden.
--------------------------------------------------------------------------------
Citation Art. 956a ZGB not found in laws_de.csv
Citation: Art. 956a Abs. 3 ZGB
Title: Schweizerisches Zivilgesetzbuch vom 10. Dezember 1907 - 1.  B

### now lets try to get these two citations by
1. making the llm generate meaningful questions 
2. retrieving in such a way it contains these two at least

In [131]:
def get_questions_from_mistral(query: str) -> list[str]:
    llm = ModalQwenChat(
    api_url="https://keshavsharma25--completion.modal.run",
    temperature=0.5,
    max_new_tokens=512,
    top_p=0.95,
    max_retries=1,
    http_timeout=600,
    system_prompt=QUERIES_GENERATE_SYSTEM_PROMPT,
)
    response = llm.invoke(query)
    questions = response.content.split("\n")[1:]  # take only the first 5 lines
    return [q.strip() for q in questions if q.strip()]

In [ ]:
for i in range(100):
    query = train_csv.iloc[i]["query"]
    gold_citations = train_csv.iloc[i]["gold_citations"].split(";")
    questions = get_questions_from_mistral(query)
    
    print("=" * 80)

→ POST to Modal (Qwen) | 1 msgs | 2,399 chars | ~799 tokens
Raw response from LLM:
Gewerbebauverordnung Änderungen
UVP Pflicht
Lärmpegel 53 dB(A)
Metallschredder Modernisierung
Baugenehmigung Anforderungen
→ POST to Modal (Qwen) | 1 msgs | 2,280 chars | ~760 tokens
Raw response from LLM:
Generate search queries based on the provided legal case description:

Grundbuchloeschung
Dienstbarkeit ungerechtfertigt
Liegenschaft an Kanalisation anschliessen
Grundbuchverwalter Entscheidung
Klage gegen Grundbuchverwalter
→ POST to Modal (Qwen) | 1 msgs | 1,999 chars | ~666 tokens
Raw response from LLM:
GENERATE:
Römer Statut Tatbestand
Römer Statut Folter
Römer Statut Zuständigkeit
Römer Statut Vernehmung
Schweiz Römer Statut Zuständigkeit
→ POST to Modal (Qwen) | 1 msgs | 3,803 chars | ~1,267 tokens
Raw response from LLM:
query: Schiedsverfahren Zuständigkeit
query: Schiedsinstitution Bestimmung
query: Unzuständigkeitseinrede Prozess
query: Entscheid Unabhängigkeit Hauptverhandlung
query: Anfecht

KeyboardInterrupt: 

### flow for single query

→ POST to Modal (Qwen) | 1 msgs | 2,399 chars | ~799 tokens
Raw response from LLM:
Geräuschbelastung
Lärmpegel
UVP Pflicht
Metallschredder Modernisierung
Bewilligung Anforderungen


In [183]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, chat_models
load_dotenv()

True

In [184]:
chat_models

<module 'langchain_google_genai.chat_models' from '/Users/keshavsharmaog/Downloads/swiss-law-agentic-retrieval/env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py'>

In [200]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview"
)
llm.invoke("hi")


AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EpICCo8CAb4+9vuLlpmUoTXFAxTeKnm3Y1AuapngbQbdadpZQMx1V90SBNJw7y+md9FTVD6irrD+ciGVt+HdXtELuNRSlAg2PjNHZzeYFaKnxAxhvBrKRR1VN3q75NWihDOWqJXf3rJhX6TrXpRNsDd9Tg9dHZkZBN9OCHxXGw3b8HP4egSBQAmI+lWNawh/U/Zq82D5COdL6PEwQly4YGGJxkrIH9dg/KeFM7cJ7+y6TTFR0WFbpyzmHgiN74hST6QcvwWqHjryAomKH+HO+Ejck5C/DUQ3fDOgSGQJfyiozTdBaBF47ARWN4fH1wLNESZS5mZdjBmA1uPBQ6lY3w+vXO2l2+kxG/OZ9xBUoRl5Z3p7WQ=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d58a5-6206-7271-99bb-b690c8746c1b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 71, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 62}})

In [201]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder("../models/cross_encoder_msmarco_finetuned", num_labels=1, device="mps")

def get_questions_from_google(query: str) -> list[str]:
    llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview"
)
    response = llm.invoke(query)
    print("Raw response:", response.content)
    questions = response.content[0]["text"].split("\n")  # take only the first 5 lines
    return [q.strip() for q in questions if q.strip()]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at ../models/cross_encoder_msmarco_finetuned and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [195]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs={"k": 1000},
)

i = 1
query = QUERIES_GENERATE_SYSTEM_PROMPT + "\n Legal question: " + train_csv.iloc[i]["query"]
gold_citations = train_csv.iloc[i]["gold_citations"].split(";")
questions = get_questions_from_google(query)
print("Generated questions:")
print(questions)

def rerank_and_filter(questions: list[str], retriever, reranker_model, top_k=10) -> list[str]:
    all_docs = []
    for question in questions:
        print(f"question: {question}")
        retrieved_docs = retriever.invoke(question)
        # now rerank
        pairs = [(question, doc.page_content) for doc in retrieved_docs]
        scores = reranker_model.predict(pairs)
        scored_docs = sorted(zip(retrieved_docs, scores), key=lambda x: x[
1], reverse=True)
        top_docs = [doc for doc, score in scored_docs[:top_k]]
        all_docs.extend(top_docs)
    # deduplicate and return citations
    unique_citations = set()
    for doc in all_docs:
        if doc.metadata.get("citation"):
            unique_citations.add(doc.metadata["citation"])
    return list(unique_citations)



citations = rerank_and_filter(questions, retriever, reranker_model, top_k=10)

count_citation_found = 0
citations = list(set(citations))
for cit in gold_citations:
    if str(cit).lower() in [str(c).lower() for c in citations]:
        count_citation_found += 1
        print(f"Found gold citation in retrieved results: {cit}")
    else:
        print(f"Gold citation NOT found in retrieved results: {cit}")
print(f"Total gold citations: {len(gold_citations)}")
print(f"Gold citations found in retrieved results: {count_citation_found}")

Raw response: [{'type': 'text', 'text': 'ungerechtfertigte Löschung dingliches Recht\nLöschung Dienstbarkeit von Amtes wegen\nKlage auf Grundbuchberichtigung\nBeschwerde Verfügung Grundbuchverwalter\nHaftung Führung Grundbuch', 'extras': {'signature': 'EtMqCtAqAb4+9vucX9/bptzi1OM1OoXpIzwBMUJehswskJJJz11p4dLLIIWz5TI3Ye4OLLecF7PN+huTCEqZZi+revzLdD0af6J3pKS4xtgmqKhnJbfLoW+cx+9u7+ESPlEUfpJYN89U4/k48ZoOVDeVeLjA5jzT6TrFJyWemgZM1ster5R0WA8It+WNG1x0wQ/4ePfoQOnUoABdQvfiqIAaDI0JX9hvCK+c0JgagQn66AyOJEFcgXjCL2YStT1tKkeY3bzz8y9tcmhO6YasC+HP633EpbWNJTkzc/xkVOaES7FjPdXa3l7CAxuNdrNQI+IrRCwozGds6OXcCnWQxlyc3MjrWVMB7vBpO0fYawiYpmb4dicGwUs3qZ6U1KToS+JUgJGdUaLHtd/Ng8tuXYdD8QwhCzLDSBfVQnaIPeKdkmCpCQ2hVPiW3uHeR8Y1fsYpDWVl8LdY8cwHskDAu/ySErRtr9wpP92nAZJ/TV9OtZ80ROAbHar7qv+JAh/9PPebrx6f2BKOST77Y0pTtEwGVCMwZ7MNHkwu+b6ZYsBqcatTmdjGBTom5geWA9HvKrEKhvaTZmZypch4BNVSq1EYtN2BYdPZNEBWObzbcvhbAqcLGKQhACuqQ7Zhm9CNZyp2ZhBnwNCw0nNpjf6XxilLxIyB/z18bPCsVD6CH+QnZYzswztinlqlyEeAF8A/qbaDf5qkOZv9DzdPHSweb03Vt0Iucpgn7r87rEzoBI3

In [196]:
citations

['Art. 138 Abs. 1 GBV',
 'Art. 19 Abs. 2 142.513',
 'Art. 93 Abs. 2 HRegV',
 'Art. 38 GBV',
 'Art. 7 Abs. 2 EKBV',
 'Art. 61 ZSV',
 'Art. 33 Abs. 2 AStG',
 'Art. 5 Abs. 1 BPDV',
 'Art. 7c Abs. 2 OV-EJPD',
 'Art. 54a Abs. 6 ELV',
 'Art. 53 Abs. 3 ZDG',
 'Art. 676 Abs. 3 ZGB',
 'Art. 5 Abs. 3 221.215.328.4',
 'Art. 130a Abs. 2 VZG',
 'Art. 4 Abs. 2 VSL',
 'Art. 52 ZDG',
 'Art. 19 Abs. 2 AIAG',
 'Art. 9 Abs. 2 172.327.1',
 'Art. 112 Abs. 2 VBVA',
 'Art. 27 Abs. 2 VGR',
 'Art. 64 Abs. 2 LFG',
 'Art. 14 Abs. 2 PBG',
 'Art. 5 Abs. 7 NDG',
 'Art. 762 ZGB',
 'Art. 112j Abs. 3 ZV',
 'Art. 25 Abs. 5 367.1',
 'Art. 55 Abs. 2bis MSchG',
 'Art. 22 Abs. 2 BVG',
 'Art. 26 EKBV',
 'Art. 67 WaV',
 'Art. 145 Abs. 1 KAG',
 'Art. 24d Abs. 1 MSchV',
 'Art. 14 Abs. 3 PVO-TVS',
 'Art. 35 Abs. 2 MinöStG',
 'Art. 95 Abs. 1 MStV',
 'Art. 9 GlG',
 'Art. 66 Abs. 2 MSchG',
 'Art. 11 Abs. 3 ZDG',
 'Art. 51 Abs. 2 AlkG',
 'Art. 971 Abs. 2 OR',
 'Art. 2 748.217.11',
 'Art. 18 Abs. 3 NDG',
 'Art. 8 Abs. 1 361.4',
 'Ar